# Unit 2: Lesson 11 | Vision-Reactive Robot

## Your final vision project

You know how to collect detection results. Today you'll make your robot react to changes instead of repeating the same response.

**Safety:** Keep the car stationary on a stable surface. Stop the motors, use approved objects, keep audio comfortable, and make sure nobody else is using the robot or vision detectors. Do not add driving code to the main project.

Run cells individually. Hardware activities are disabled until you deliberately enable them with instructor permission.

## 1. Plan your behavior

Choose a colored object (or a supported object category if your model is available). Fill in this plan before coding.

| Decision | Your plan |
|---|---|
| Target and detector | |
| Response on appearance | |
| Response while still detected | |
| Response on disappearance | |
| Response if data is unavailable | |
| Observation limit | |

Keep the first response short and stationary.

## 2. Prepare your tools

Run this setup cell once. The helpers are provided so you can focus on the project. `RUN_HARDWARE` controls the real robot exercises. Leave it `False` while testing simulated results.

In [ ]:
import time
import math

RUN_HARDWARE = False


def finish_audio(car):
    try:
        car.stop_audio()
    finally:
        car.close()


def with_audio(task):
    from betabox_robotics import BetaboxCar

    car = BetaboxCar()
    try:
        car.stop()
        task(car)
    finally:
        finish_audio(car)

### Test a Boolean response

Complete `react_to_boolean()` so it speaks a short message when `detected` is true and prints a searching message otherwise. With permission, enable hardware and run the cell twice, changing `detected` each time. Turn hardware off again between exercises if needed.

In [ ]:
def react_to_boolean(car, detected):
    # TODO: Use if/else to speak or print a message.
    pass


detected = True
if RUN_HARDWARE:
    with_audio(lambda car: react_to_boolean(car, detected))
else:
    print("Audio activity disabled.")

## 3. Verify actual detections

The service keeps the latest detector result. `fresh_result()` waits for a newer timestamp and refuses error results. An empty detection list in a fresh, successful result is valid.

`with_detector()` opens one session, checks that the requested detector is not already enabled, enables it, and attempts to disable it and close the car even if an error occurs. Do not use it alongside another vision activity. These are shared detectors, not privately owned by the notebook.

Read the helper, then run the next cell when your instructor approves. The waiting budget limits polling, not the duration of a blocking hardware or HTTP request.

In [ ]:
def fresh_result(car, source, after, timeout=8.0):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        result = car.metadata(source)
        if (result is not None and result.source == source
                and math.isfinite(result.timestamp)
                and result.timestamp > after):
            if result.data.get("error"):
                raise RuntimeError(str(result.data["error"]))
            return result
        time.sleep(0.2)
    raise TimeoutError("No fresh usable detection result.")


def target_detected(result, target):
    return any(d.label.casefold() == target.casefold()
               for d in result.detections)


def with_detector(source, target, task):
    from betabox_robotics import BetaboxCar

    car = BetaboxCar()
    attempted_enable = False
    try:
        car.stop()
        status = car.detection_status()
        if source not in status.detectors:
            raise RuntimeError(f"Detector unavailable: {source}")
        if status.is_enabled(source):
            raise RuntimeError("Detector already enabled. Ask your instructor.")
        attempted_enable = True
        if source == "color":
            car.enable_color_detection(target)
        elif source == "objects":
            car.enable_detection(source)
        else:
            raise ValueError("Choose color or objects.")
        task(car, time.time())
    finally:
        try:
            if attempted_enable:
                if source == "color":
                    car.disable_color_detection()
                else:
                    car.disable_detection(source)
        finally:
            finish_audio(car)

In [ ]:
source = "color"
target = "red"
# For object detection, use source = "objects" and a supported label.


def observe(car, after):
    result = fresh_result(car, source, after)
    print("Labels:", [d.label for d in result.detections])
    print("Target detected:", target_detected(result, target))


if RUN_HARDWARE:
    with_detector(source, target, observe)

Run the observation with your target visible, then remove it and run again. Try a different position or background. Record your results here.

| Scene | Detected? | Notes |
|---|---|---|
| Target visible | | |
| Target removed | | |
| Different position | | |

If you receive an error, don't interpret it as a missing target. Ask your instructor if it persists. If the object model is unavailable, use color detection instead.

## 4. Recognize a change

Predict how many times this program will print `Appeared`. Run it and change the sequence. Notice that the previous value is updated only after the condition is evaluated.

In [ ]:
detections = [False, False, True, True, True, False, True]
previously_detected = False

for detected in detections:
    if detected and not previously_detected:
        print("Appeared")
    elif not detected:
        print("Searching")
    previously_detected = detected

### Your turn: create four responses

Complete `choose_event()` so it returns `appeared`, `lost`, `hold`, or `search`. Return `unavailable` when the current result is `None`. Run the test cell until every case passes. This exercise uses no hardware.

For an unavailable result, finish rather than changing the stored state as if the target disappeared.

In [ ]:
def choose_event(detected, previously_detected):
    if detected is None:
        return "unavailable"
    if detected and not previously_detected:
        return "appeared"
    # TODO: Recognize a lost target, a held target, and searching.
    return "TODO"


cases = [
    (False, False, "search"),
    (True, False, "appeared"),
    (True, True, "hold"),
    (False, True, "lost"),
    (None, False, "unavailable"),
    (None, True, "unavailable"),
]
for current, previous, expected in cases:
    actual = choose_event(current, previous)
    print(current, previous, actual,
          "PASS" if actual == expected else f"Expected {expected}")

### Simulate an entire audio sequence

Complete `perform_event()` so `appeared` plays one short note or speaks one message, `lost` prints `Target lost!`, and the other states never repeat the sound. Test first with hardware disabled, then with permission enable both flags. Try several consecutive `True` values and two separate appearances. The missing result should end the loop without a false lost message.

In [ ]:
RUN_SIMULATED_AUDIO = False


def perform_event(car, event):
    # TODO: Choose a short sound on appeared and print on lost.
    # Only call audio methods when car is not None.
    # The simulation prints the event when no car is connected.
    if car is None:
        print(event)


def simulate(car=None):
    previously_detected = False
    for detected in [False, True, True, False, True, None, True]:
        event = choose_event(detected, previously_detected)
        if event == "unavailable":
            print("Detection unavailable. Ending test.")
            break
        perform_event(car, event)
        previously_detected = detected


if RUN_HARDWARE and RUN_SIMULATED_AUDIO:
    with_audio(simulate)
else:
    simulate()

## 5. Final project: Vision-Reactive Robot

Complete the starter below using your finished functions. The helper supplies a fresh result and manages cleanup. You must convert each result to a Boolean, choose the event, perform the response, and update the remembered state.

- Start with the target out of view.
- Use your chosen short stationary response when the target appears.
- Report when it is lost; do not repeat audio during `hold`.
- On a failed observation, report the problem and finish without announcing a loss.
- Limit the program to 20 observations. The provided 45-second loop budget is checked between operations, not a hard deadline.

Set `RUN_PROJECT` to `True` only when your logic passes the simulation and your instructor approves. Keep the drive motors stopped.

In [ ]:
RUN_PROJECT = False
MAX_CHECKS = 20
MAX_SECONDS = 45


def run_project(car, after):
    from betabox_robotics.vision.client import VisionClientError

    previously_detected = False
    deadline = time.monotonic() + MAX_SECONDS
    for check in range(MAX_CHECKS):
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            print("Time budget reached.")
            break
        try:
            result = fresh_result(car, source, after, timeout=min(8.0, remaining))
        except (VisionClientError, RuntimeError, TimeoutError) as error:
            print("Detection unavailable:", error)
            break
        after = result.timestamp
        if time.monotonic() >= deadline:
            print("Time budget reached.")
            break

        # TODO: Determine whether the target is detected.
        # TODO: Choose the event using the previous state.
        # TODO: Perform the response.
        # TODO: Update previously_detected AFTER responding.

        time.sleep(0.3)
    print("Observation session finished.")


if RUN_HARDWARE and RUN_PROJECT:
    with_detector(source, target, run_project)
else:
    print("Final project disabled.")

### Test and improve

Record actual results, making one change at a time.

| Test | Expected behavior | Actual result / improvement |
|---|---|---|
| Target enters view | Respond once | |
| Target remains visible | No repeated audio | |
| Target leaves | Report loss | |
| Target reappears | Respond again | |
| Simulated unavailable result | Finish without false loss | |
| Check or time budget ends | Finish and release resources | |

### Extra challenge

Choose one: respond to two colors, count appearances, show a detected center, or require three consecutive valid observations before changing state. Reset a confirmation count when a different result arrives. Keep the main project stationary. Any movement extension requires separate instructor approval and stand testing.

## Before you finish

- Complete the decision tests and final project.
- Confirm that your detector is disabled and the car session is closed. If cleanup reports an error, ask your instructor for help.
- Set your hardware flags back to `False` and save your notebook.
- Return to **Unit 2, Lesson 11** on the Betabox webpage to reflect on your project.

You've reached the end of Unit 2. You can now combine programming, movement, sensors, sound, and vision in original robotics projects.